In [10]:
# Cell 1: config + browser startup.
import importlib
import json
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "browser").exists():
    ROOT = ROOT.parent.resolve()
if not (ROOT / "browser").exists():
    raise RuntimeError("Could not locate repo root.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

browser_driver = importlib.import_module("browser.driver")
core_config = importlib.import_module("core.config")
service_linkedin = importlib.import_module("services.linkedin.linkedin")
embedded_mongo = importlib.import_module("storage.embedded_mongo")
linkedin_runtime = importlib.import_module("tests.llm_test_linkedin.linkedin_runtime")

browser_driver = importlib.reload(browser_driver)
core_config = importlib.reload(core_config)
service_linkedin = importlib.reload(service_linkedin)
embedded_mongo = importlib.reload(embedded_mongo)
linkedin_runtime = importlib.reload(linkedin_runtime)

from browser.driver import build_driver
from core.config import load_app_config

app_config = load_app_config()
browser_cfg = dict(app_config["profile"]["browser"])
old_user_data_dir = Path(r"D:\_Desktop\Projects\Automations prj\User Data")
if old_user_data_dir.exists():
    browser_cfg["user_data_dir"] = str(old_user_data_dir)

store = embedded_mongo.EmbeddedMongoStore(Path(app_config["profile"]["mongo_file"]))
driver = build_driver(browser_cfg)
LLM_TEST_DIR = ROOT / "tests" / "llm_test_linkedin"

runtime_state = {
    "driver": driver,
    "store": store,
    "log_path": "mock://mongodb/linkedin",
    "api_key_path": str(Path(r"D:\_Desktop\api_key.txt")),
    "model_backend": "local",
    "openai_model": "gpt-5.4-mini",
    "openai_base_url": "https://api.openai.com/v1/chat/completions",
    "openai_reasoning_effort": "low",
    "llama_model": "qwen3.5-9b",
    "llama_base_url": "http://127.0.0.1:8080/v1/chat/completions",
    "max_completion_tokens": 1024,
    "timeout": 300,
    "retries": 2,
    "context_tokens": 20000,
    "response_reserve_tokens": 2048,
    "delays": {
        "open_jobs_search_page": 1,
        "set_keyword_input": 1,
        "set_location_input": 1,
        "open_all_filters_menu": 1,
        "sync_filters_state": 0.2,
        "show_results": 1,
        "go_to_page": 3,
        "click_listing_card": 1,
        "click_listing_card_jitter": 0.2
    },
    "verbose": False,
    "session_outputs": [],
    "current_linkedin_state": {},
    "last_listing_payload": {},
    "last_detail_payload": {},
}

bundle = linkedin_runtime.load_instruction_bundle(LLM_TEST_DIR)
print(json.dumps({"root": str(ROOT), "browser": browser_cfg, "backend": runtime_state["model_backend"]}, indent=2))


config: dir D:\_Desktop\Projects\Automations prj\Job_search\JobSeekr_fresh\config
config: load profile.json
config: load extract.json
config: load presets.json
store: open runtime\mongo_store.json
driver: start
driver: launch chrome
driver: ready
{
  "root": "D:\\_Desktop\\Projects\\Automations prj\\Job_search\\JobSeekr_fresh",
  "browser": {
    "headless": false,
    "use_stealth": true,
    "version_main": 149,
    "page_load_timeout_seconds": 45,
    "wait_timeout_seconds": 20,
    "start_maximized": true,
    "user_data_dir": "D:\\_Desktop\\Projects\\Automations prj\\User Data"
  },
  "backend": "local"
}


In [11]:
# Cell 2: runtime loop.
import importlib
from typing import Any

service_linkedin = importlib.import_module("services.linkedin.linkedin")
linkedin_runtime = importlib.import_module("tests.llm_test_linkedin.linkedin_runtime")
llm_runtime = importlib.import_module("tests.llm_test.llm_runtime")
service_linkedin = importlib.reload(service_linkedin)
llm_runtime = importlib.reload(llm_runtime)
linkedin_runtime = importlib.reload(linkedin_runtime)

from tests.llm_test_linkedin.linkedin_runtime import run_agent as run_linkedin_agent

def run_agent(task: str, max_steps: int = 20) -> dict[str, Any]:
    return run_linkedin_agent(
        task,
        runtime_state,
        bundle,
        backend=runtime_state.get("model_backend", "local"),
        max_steps=max_steps,
    )


In [ ]:
# Cell 3: run a task example.
# Edit the task string below, then run this cell.

task = "Find entry-level software engineering jobs in United States. Use filters when they help and fetch pages 1-3, then open the 2 most promising listings and compare them."
result = run_agent(task)
result
